## Audio books analysis
This model will analyse the data of past clients to train and then be able to forecast whether a customer is likely to buy again or not.

In [12]:
# import libraries
import pandas as pd  # to read the CSV
import numpy as np
import tensorflow as tf


In [29]:
#Load data from csv
abooks_df = pd.read_csv("../../../../statistics/python/audiobooks/Audiobooks_data.csv",header=None)
abooks_df.head()
#ID,Book length(mins)_overal,Book length (mins)_avg,Price_overall,Price_avg,Review,Review 10/10,Minutes listened,Completion,Support Requests,Last visited minus Purchase date,Targets
#0       1                               2                3           4        5            6           7              8              9                10                         11

abooks_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14084 entries, 0 to 14083
Data columns (total 12 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       14084 non-null  int64  
 1   1       14084 non-null  float64
 2   2       14084 non-null  int64  
 3   3       14084 non-null  float64
 4   4       14084 non-null  float64
 5   5       14084 non-null  int64  
 6   6       14084 non-null  float64
 7   7       14084 non-null  float64
 8   8       14084 non-null  float64
 9   9       14084 non-null  int64  
 10  10      14084 non-null  int64  
 11  11      14084 non-null  int64  
dtypes: float64(6), int64(6)
memory usage: 1.3 MB


In [59]:
# Shuffle and separate targets from data
# Shuffle -> no shuffle for df

targets = abooks_df.iloc[:,11]
targets.info()

<class 'pandas.core.series.Series'>
RangeIndex: 14084 entries, 0 to 14083
Series name: 11
Non-Null Count  Dtype
--------------  -----
14084 non-null  int64
dtypes: int64(1)
memory usage: 110.2 KB


In [63]:
abooks_data = abooks_df.iloc[:,0:11]
abooks_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14084 entries, 0 to 14083
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       14084 non-null  int64  
 1   1       14084 non-null  float64
 2   2       14084 non-null  int64  
 3   3       14084 non-null  float64
 4   4       14084 non-null  float64
 5   5       14084 non-null  int64  
 6   6       14084 non-null  float64
 7   7       14084 non-null  float64
 8   8       14084 non-null  float64
 9   9       14084 non-null  int64  
 10  10      14084 non-null  int64  
dtypes: float64(6), int64(5)
memory usage: 1.2 MB


In [64]:
#Normalize columns in the dataframe
def scale_float(data_in):
    data_float = data_in.astype(np.float64)
    return((data_float-data_float.mean())/data_float.std())

abooks_df_normal = abooks_data.apply(scale_float, 0, True, engine='numba', engine_kwargs={'parallel': True})
abooks_df_normal

,0,1,2,3,4,5,6,7,8,9,10
0,-1.628081,0.056944,-0.089504,2.560319,2.191789,2.284918,1.694489,3.583548,3.810353,10.441350,0.340855
1,-1.612707,1.127687,0.735156,-0.359686,-0.398171,-0.437653,0.000319,-0.520980,-0.511732,-0.148730,-0.702175
2,-1.518191,1.127687,0.735156,-0.359686,-0.398171,-0.437653,0.000319,-0.520980,-0.511732,-0.148730,3.696693
3,-1.433271,0.056944,-0.089504,-0.231936,-0.284861,-0.437653,0.000319,1.220335,1.321880,1.969286,0.760335
4,-1.385806,1.127687,0.735156,-0.359686,-0.398171,-0.437653,0.000319,0.391137,0.768886,-0.148730,3.390586
...,...,...,...,...,...,...,...,...,...,...,...
14079,1.181195,0.056944,-0.089504,-0.359686,-0.398171,2.284918,0.140205,2.008072,2.151371,-0.148730,-0.656826
14080,1.227731,-1.013799,-0.914164,-0.112297,-0.178744,2.284918,-4.522648,0.681356,0.332311,-0.148730,-0.373394
14081,1.481872,1.127687,0.735156,-0.195436,-0.252486,-0.437653,0.000319,-0.520980,-0.511732,-0.148730,-0.702175
14082,1.657078,0.056944,-0.089504,-0.359686,-0.398171,2.284918,-1.414080,1.054495,1.147250,-0.148730,0.318181


In [65]:
requests = abooks_df.iloc[:,9]
requests.mean()

np.float64(0.0702215279750071)

In [66]:
requests.max()

np.int64(30)

In [67]:
abooks_df_normal.loc[abooks_df[9]==requests.max(),:]

,0,1,2,3,4,5,6,7,8,9,10
8309,0.583453,1.127687,0.735156,-0.359686,-0.398171,2.284918,1.694489,3.210409,4.727159,63.391751,-0.237347


In [68]:
abooks_df.iloc[8309,:]

0     22427.00
1      2160.00
2      2160.00
3         5.33
4         5.33
5         1.00
6        10.00
7         0.90
8      1944.00
9        30.00
10       41.00
11        0.00
Name: 8309, dtype: float64

In [69]:
abooks_df_normal.iloc[:,4].std()

np.float64(1.0000355031687493)

In [70]:
abooks_np = np.array(abooks_df_normal)
abooks_np

array([[-1.62808146,  0.05694432, -0.08950406, ...,  3.81035339,
        10.44134984,  0.34085525],
       [-1.61270711,  1.12768719,  0.7351559 , ..., -0.51173244,
        -0.14873032, -0.70217541],
       [-1.51819094,  1.12768719,  0.7351559 , ..., -0.51173244,
        -0.14873032,  3.69669302],
       ...,
       [ 1.48187206,  1.12768719,  0.7351559 , ..., -0.51173244,
        -0.14873032, -0.70217541],
       [ 1.6570778 ,  0.05694432, -0.08950406, ...,  1.14725   ,
        -0.14873032,  0.31818067],
       [-1.70474687,  0.1640186 ,  2.5494078 , ..., -0.51173244,
        -0.14873032, -0.70217541]])

In [71]:
# 10% of the data to be used as validation and 10% as validation
num_validation_samples = tf.cast(abooks_np.shape[0]*0.1, tf.int64)

In [74]:
test_data = abooks_np.iloc[0:num_validation_samples,:]
test_data.shape()

AttributeError: 'numpy.ndarray' object has no attribute 'iloc'

## My take at the preprocessing

In [103]:
import numpy as np
from sklearn import preprocessing

### Load the data

In [104]:
raw_csv_data = np.loadtxt('../../../../statistics/python/audiobooks/Audiobooks_data.csv', delimiter=',')
unscaled_inputs_all = raw_csv_data[:,1:-1]
targets_all = raw_csv_data[:,-1]

### Balance the dataset

In [113]:
#Obtain the different classes from the targets column
target_classes,counts=np.unique(targets_all, return_counts=True)
# number of samples in the smallest data class, to limit all other classes to this number of samples
num_samples_one_class = counts.min()
print(num_samples_one_class)
samples_indices_dict = {}  #will contain the "counts.min()" indices for each of the target classes
for i in target_classes:    # this approach takes the first 'num_samples_one_class' of each class, not really shuffled.
                            # it later shuffles the selected ones alone
    class_filter = target_all[:] == i
    samples_indices_dict[i] = np.where(class_filter)[0][:num_samples_one_class]

balanced_indices = np.concatenate(list(samples_indices_dict.values()))

2237


### Shuffle the data

In [106]:
#shuffle the data by shuffling the indices and then "slice" data and targets with the indices
shuffled_indices = np.copy(balanced_indices)
np.random.shuffle(shuffled_indices)
shuffled_targets = targets_all[shuffled_indices]
shuffled_inputs = unscaled_inputs_all[shuffled_indices]

### Standardize the inputs

In [107]:
#Standardize the inputs
scaled_inputs = preprocessing.scale(shuffled_inputs)

### Split in train, validate and test

In [112]:
num_samples = scaled_inputs.shape[0]
num_train_samples = int(num_samples * 0.8)
num_validate_samples = int (num_samples * 0.1)
num_test_samples = num_samples - num_train_samples - num_validate_samples

train_inputs = scaled_inputs[:num_train_samples,:]
train_targets = shuffled_targets[:num_train_samples]

validate_inputs = scaled_inputs[num_train_samples:num_train_samples + num_validate_samples,:]
validate_targets = shuffled_targets[num_train_samples:num_train_samples + num_validate_samples]

test_inputs = scaled_inputs[num_train_samples + num_validate_samples:,:]
test_targets = shuffled_targets[num_train_samples + num_validate_samples:]

# Check that the '0'- and '1'-classes are balanced in each of the three datasets
print(np.sum(train_targets), num_train_samples, np.sum(train_targets)/num_train_samples)
print(np.sum(validate_targets), num_validate_samples, np.sum(validate_targets)/num_validate_samples)
print(np.sum(test_targets), num_test_samples, np.sum(test_targets)/num_test_samples)


1788.0 3579 0.49958088851634536
233.0 447 0.5212527964205816
216.0 448 0.48214285714285715


# Gemini's take at preprocessing the Audiobook dataset

In [120]:
import numpy as np

# 1. Load and Strip ID
raw_csv_data = np.loadtxt('../../../../statistics/python/audiobooks/Audiobooks_data.csv', delimiter=',')
# Remove first column (ID) and separate Features (X) from Labels (y)
all_features = raw_csv_data[:, 1:-1]
all_labels = raw_csv_data[:, -1]

# 2. Identify and Count Classes
unique_classes, counts = np.unique(all_labels, return_counts=True)
minority_class_size = np.min(counts)

print(f"Classes found: {unique_classes}")
print(f"Balancing all classes to {minority_class_size} samples each.")

# 3. Balanced Index Selection (The Generalized Part)
balanced_indices = []

for c in unique_classes:
    # Get indices where the label matches class 'c'
    class_indices = np.where(all_labels == c)[0]
    
    # Shuffle these specific indices so we pick randomly
    np.random.shuffle(class_indices)
    
    # Take only as many as the minority class has
    balanced_indices.append(class_indices[:minority_class_size])

# Combine all selected indices into one flat array
balanced_indices = np.concatenate(balanced_indices)

# 4. Final Shuffle
# This mixes the classes together so they aren't grouped
np.random.shuffle(balanced_indices)

# 5. Apply to data
X_balanced = all_features[balanced_indices]
y_balanced = all_labels[balanced_indices]

Classes found: [0. 1.]
Balancing all classes to 2237 samples each.


### Split dataset in 80/10/10 for train/validate/test

In [121]:
# 1. Determine the total count of balanced samples
samples_count = balanced_indices.shape[0]

# 2. Calculate the split points (cumulative)
# 80% for training
train_samples_count = int(0.8 * samples_count)
# 10% for validation (added to the 80% to find the index)
validation_samples_count = int(0.1 * samples_count)

# 3. Slice the data using the shuffled balanced indices
# Training set (from 0 to 80%)
train_inputs = all_features[balanced_indices[:train_samples_count]]
train_targets = all_labels[balanced_indices[:train_samples_count]]

# Validation set (from 80% to 90%)
validation_inputs = all_features[balanced_indices[train_samples_count : train_samples_count + validation_samples_count]]
validation_targets = all_labels[balanced_indices[train_samples_count : train_samples_count + validation_samples_count]]

# Test set (the remaining 10%)
test_inputs = all_features[balanced_indices[train_samples_count + validation_samples_count:]]
test_targets = all_labels[balanced_indices[train_samples_count + validation_samples_count:]]

# 4. Verify the distribution (Optional but recommended)
print(f"Training: {train_inputs.shape[0]} samples")
print(f"Validation: {validation_inputs.shape[0]} samples")
print(f"Test: {test_inputs.shape[0]} samples")

# 5. Save as .npz for tomorrow's class
#np.savez('Data_train', inputs=train_inputs, targets=train_targets)
#np.savez('Data_validation', inputs=validation_inputs, targets=validation_targets)
#np.savez('Data_test', inputs=test_inputs, targets=test_targets)

Training: 3579 samples
Validation: 447 samples
Test: 448 samples


In [122]:
# Check that the '0'- and '1'-classes are balanced in each of the three datasets
print(np.sum(train_targets), train_inputs.shape[0], np.sum(train_targets)/train_inputs.shape[0])
print(np.sum(validation_targets), validation_inputs.shape[0], np.sum(validation_targets)/validation_inputs.shape[0])
print(np.sum(test_targets), test_inputs.shape[0], np.sum(test_targets)/test_inputs.shape[0])


1792.0 3579 0.5006985191394244
223.0 447 0.4988814317673378
222.0 448 0.4955357142857143


## Audiobook analysis from the course

In [91]:
import numpy as np
from sklearn import preprocessing

### Load the data

In [92]:
raw_csv_data = np.loadtxt('../../../../statistics/python/audiobooks/Audiobooks_data.csv', delimiter=',')

unscaled_inputs_all = raw_csv_data[:,1:-1]
targets_all = raw_csv_data [:,-1]

### Balance the dataset

In [98]:
# there are too few '1' in the targets column, so we'll discard some '0' rows to make the dataset
#   contain as many '0' as '1'
num_one_targets = int(np.sum(targets_all))
zero_targets_counter = 0
indices_to_remove = []

for i in range(targets_all.shape[0]):
    if targets_all[i] ==0:
        # only for '0'-class targets
        zero_targets_counter +=1
        if zero_targets_counter > num_one_targets:
            indices_to_remove.append(i)

unscaled_inputs_balanced = np.delete(unscaled_inputs_all, indices_to_remove, axis=0)
targets_balanced = np.delete(targets_all, indices_to_remove, axis=0)

### Standardize the inputs

In [99]:
scaled_inputs = preprocessing.scale(unscaled_inputs_balanced)

### Shuffle the data

In [100]:
shuffled_indices = np.arange(scaled_inputs.shape[0])
np.random.shuffle(shuffled_indices)

shuffled_inputs = scaled_inputs[shuffled_indices]
shuffled_targets = targets_balanced[shuffled_indices]

### Split data set in train, validate and test.

In [101]:
samples_count = shuffled_inputs.shape[0]
train_samples_count = int(0.8*samples_count)
validation_samples_count = int(0.1*samples_count)
test_samples_count = samples_count - train_samples_count - validation_samples_count

train_inputs = shuffled_inputs[:train_samples_count]
train_targets = shuffled_targets[:train_samples_count]

validation_inputs = shuffled_inputs[train_samples_count:train_samples_count+validation_samples_count]
validation_targets = shuffled_targets[train_samples_count:train_samples_count+validation_samples_count]

test_inputs = shuffled_inputs[train_samples_count+validation_samples_count:]
test_targets = shuffled_targets[train_samples_count+validation_samples_count:]

# Check that the '0'- and '1'-classes are balanced in each of the three datasets
print(np.sum(train_targets), train_samples_count, np.sum(train_targets)/train_samples_count)
print(np.sum(validation_targets), validation_samples_count, np.sum(validation_targets)/validation_samples_count)
print(np.sum(test_targets), test_samples_count, np.sum(test_targets)/test_samples_count)


1779.0 3579 0.49706621961441744
230.0 447 0.5145413870246085
228.0 448 0.5089285714285714


### Save the three datasets in *.npz

In [11]:
np.savez('../../../../statistics/python/audiobooks/Audiobooks_data_train', inputs=train_inputs, targets=train_targets)
np.savez('../../../../statistics/python/audiobooks/Audiobooks_data_validation', inputs=validation_inputs, targets=validation_targets)
np.savez('../../../../statistics/python/audiobooks/Audiobooks_data_test', inputs=test_inputs, targets=test_targets)

### Machine learing part
Preprocesing is done. It can be reused for other problems.
From here on, we will start from the .npz files.